# Apache Arrow Flight

![Arrow Logo](images/arrow.png)

Arrow is a foundational technology in the Data Engineering space. It powers all your favourite tools
 from Spark to Polars, Duckdb and Snowflake. As they say, it's a standard

![XKDC Standards](images/Standards.png)

Why this particular standard matters, is that it solves the problem of interprocess communication
 between different languages and frameworks.

## Process interop

Take the scenario of the Spark UDF in the old days:

![ipc](images/spark_udf.png)

Without a standard for memory layout for the data, we need to introduce glue code between the JVM
Spark Memory and the Pandas Numpy memory layout, usually having to copy the data back and forth.


When we introduce Apache Arrow, both runtimes can share the same memory layout, and we can avoid
the need for any glue code or copying.
![ipc](images/spark_arrow.png)


This is true for any library which uses Arrow, like Duckdb, Pandas and Polars.

![I made this](images/i_made_this.jpg)


## Arrow as the data interchange format
Having Arrow as an universal data interchange format allows libraries to delegate responsibility for their memory layout and compute to 
Arrow and instead focus on their value-adding layer. 

We've seen this before - compilers came along, and gave us all these different optimizations and now we no longer inline statements or unravel loops. 

LLVM and JVM are both examples of the power of separating the layers of a program, so that we can focus on the value-adding part.

So Arrow is great - but what is Arrow Flight then?

# Why Flight? - in action

Let's compare to a normal REST service, which might be more familiar to more of you.

In both scenarios, we have 10,000,000 rows we want to be able to fetch, and we
want to convert the result to a dataframe.

## REST
A standard FastAPI-based REST API that you've probably written hundreds of.

Guesses? Problems?

In [1]:
import httpx2
import polars as pl

In [2]:
%%timeit -r 1
req = httpx2.get("http://rest:8000/data/db/messages")
data = req.json()
pl.from_records(data)

ReadTimeout: timed out

## Why do we care about Arrow Flight?

In [45]:
from flight_tutorial.flight_server.server import Server  # noqa
from pyarrow import flight

In [4]:
client = flight.connect("grpc://server:7000")
info = client.get_flight_info(flight.FlightDescriptor.for_path("messages"))

In [5]:
%%timeit -r 1
reader = client.do_get(info.endpoints[0].ticket)
pl.from_arrow(reader.read_all())

3.55 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


## Apache Arrow, but for servers

In essence, Arrow Flight tries to solve the same issue as Arrow, but at the Client/Server level instead of
just between processes on the same machine. We have the same problem of needing to copy the data
between DB wire protocol, ODBC and Pandas Numpy.

![DB ODBC](images/db_odbc.png)


Or in the REST example, backend format → JSON → Pandas.

![JSON API](images/api_json.png)

To make it even worse, both of these data representations are row-based, and we are targeting a column-based data format.

![Table Data](images/table_data.png)

# But wait, there's more! We also get free stuff!

Overall, a little bit more boilerplate if you're not used to Arrow Flight, but let's look at
what extras we got in the exchange, apart from the speedup.

Firstly, Arrow Flight provides great discoverability and metadata before we commit to loading data.

Let's have a closer look at what the `info` object can give us

In [6]:
info.schema

id: int64
message_id: string
campaign_id: int64
message_type: string
client_id: int64
channel: string
category: null
platform: string
email_provider: string
stream: string
date: date32[day]
sent_at: timestamp[ms]
is_opened: bool
opened_first_time_at: timestamp[ms]
opened_last_time_at: timestamp[ms]
is_clicked: bool
clicked_first_time_at: timestamp[ms]
clicked_last_time_at: timestamp[ms]
is_unsubscribed: bool
unsubscribed_at: timestamp[ms]
is_hard_bounced: bool
hard_bounced_at: timestamp[ms]
is_soft_bounced: bool
soft_bounced_at: timestamp[ms]
is_complained: bool
complained_at: timestamp[ms]
is_blocked: bool
blocked_at: timestamp[ms]
is_purchased: bool
purchased_at: timestamp[ms]
created_at: timestamp[ns]
updated_at: timestamp[ns]

The schema of the data is readily available. Did we get the correct dataset?
Does it have what I need?

In [7]:
print(
    f"Number of records: {info.total_records:,}\nSize on disk: {info.total_bytes / 1024**2:,.2f} MB"
)

Number of records: 10,000,000
Size on disk: 540.20 MB


We can see how many rows we're about to get, and the amount of data that will be
transferred over the network

In [8]:
import json

json.loads(info.app_metadata.decode("utf-8"))

{'description': 'Contains a list of all messages sent with its statuses and meta info.'}

The server can choose to send arbitrary metadata, which we can inspect here.
In this case, we get a nice description of the dataset.

## Discoverability

Discoverability is a key feature of Arrow Flight. For example, we can easily list all the datasets
we have available.

In [9]:
available_datasets = list(client.list_flights())
available_datasets

[<pyarrow.flight.FlightInfo schema=id: int64
 message_id: string
 campaign_id: int64
 message_type: string
 client_id: int64
 channel: string
 category: null
 platform: string
 email_provider: string
 stream: string
 date: date32[day]
 sent_at: timestamp[ms]
 is_opened: bool
 opened_first_time_at: timestamp[ms]
 opened_last_time_at: timestamp[ms]
 is_clicked: bool
 clicked_first_time_at: timestamp[ms]
 clicked_last_time_at: timestamp[ms]
 is_unsubscribed: bool
 unsubscribed_at: timestamp[ms]
 is_hard_bounced: bool
 hard_bounced_at: timestamp[ms]
 is_soft_bounced: bool
 soft_bounced_at: timestamp[ms]
 is_complained: bool
 complained_at: timestamp[ms]
 is_blocked: bool
 blocked_at: timestamp[ms]
 is_purchased: bool
 purchased_at: timestamp[ms]
 created_at: timestamp[ns]
 updated_at: timestamp[ns] descriptor=<pyarrow.flight.FlightDescriptor path=[b'messages']> endpoints=[<pyarrow.flight.FlightEndpoint ticket=<pyarrow.flight.Ticket ticket=b'{"name":"messages","bucket":"events","file_name":

### Listing with criteria

We can also apply send `criteria`, basically a set of bytes the server has implemented. In this case, the server has
implemented a matching search, so we can ask for all datasets that contain the string "mess"

In [46]:
list(client.list_flights(b"mess"))

[<pyarrow.flight.FlightInfo schema=id: int64
 message_id: string
 campaign_id: int64
 message_type: string
 client_id: int64
 channel: string
 category: null
 platform: string
 email_provider: string
 stream: string
 date: date32[day]
 sent_at: timestamp[ms]
 is_opened: bool
 opened_first_time_at: timestamp[ms]
 opened_last_time_at: timestamp[ms]
 is_clicked: bool
 clicked_first_time_at: timestamp[ms]
 clicked_last_time_at: timestamp[ms]
 is_unsubscribed: bool
 unsubscribed_at: timestamp[ms]
 is_hard_bounced: bool
 hard_bounced_at: timestamp[ms]
 is_soft_bounced: bool
 soft_bounced_at: timestamp[ms]
 is_complained: bool
 complained_at: timestamp[ms]
 is_blocked: bool
 blocked_at: timestamp[ms]
 is_purchased: bool
 purchased_at: timestamp[ms]
 created_at: timestamp[ns]
 updated_at: timestamp[ns] descriptor=<pyarrow.flight.FlightDescriptor path=[b'messages']> endpoints=[<pyarrow.flight.FlightEndpoint ticket=<pyarrow.flight.Ticket ticket=b'{"name":"messages","bucket":"events","file_name":

In [11]:
Server.list_flights??

Signature:
Server.list_flights(
    self,
    context: pyarrow._flight.ServerCallContext,
    criteria: bytes,
) -> Iterator[pyarrow._flight.FlightInfo]
Source:   
    def list_flights(
        self, context: flight.ServerCallContext, criteria: bytes
    ) -> Iterator[flight.FlightInfo]:
        """Flight has native support for data discovery. The client can ask for all available
         flights and can send criteria to filter, where the criteria implementation is up to
        the implementer.
        """
        with self._dataset_repo as repo:
            datasets = repo.get_datasets(name_filter=criteria.decode("utf-8"))
            for dataset in datasets:
                yield self._make_flight_info(dataset)
File:      /app/.venv/lib/python3.14/site-packages/flight_tutorial/flight_server/server.py
Type:      function

In [12]:
Server._make_flight_info??

Signature:
Server._make_flight_info(
    self,
    dataset: flight_tutorial.flight_server.models.Dataset,
) -> pyarrow._flight.FlightInfo
Source:   
    def _make_flight_info(self, dataset: Dataset) -> flight.FlightInfo:
        """
        FlightInfo is the metadata for a dataset. We store the metadata in a database,
        but that is entirely optional.
        """
        data = pq.ParquetFile(dataset.location, filesystem=self._fs)
        endpoints = [
            flight.FlightEndpoint(
                dataset.model_dump_json().encode("utf-8"),
                [self._location, *self._workers],
            )
        ]
        return flight.FlightInfo(
            schema=data.schema_arrow,
            descriptor=flight.FlightDescriptor.for_path(dataset.name),
            endpoints=endpoints,
            total_records=dataset.num_rows,
            total_bytes=dataset.serialized_size,
            app_metadata=json.dumps({"description": dataset.description}),
        )
File:      /app/

### FlightDescriptor - Asking for a specific dataset

To get a specific dataset, we can use a FlightDescriptor, basically a human-readable way of
communicating the dataset we want.

There are two types of FlightDescriptor, the `path` and the `command`.

Semantically, `path` relates to a location of data such as the name of a table or file.
`command` is a more implementation-specific approach, where the server can interpret the command
and execute it. This allows us to implement some powerful DSLs on top of Flight as needed.

Note that like REST and HTTP verbs (like GET, POST, PATCH), we, as implementors of a Flight server,
need to think about what the semantics are, but Arrow Flight doesn't impose any restrictions.

Let's go through the download process again, step-by-step.


## FlightInfo

First we need to get the `FlightInfo` about the dataset. We know that our data has the path
`message`, so we create a `FlightDescriptor` for that path

In [13]:
info = client.get_flight_info(flight.FlightDescriptor.for_path("messages"))

In [14]:
Server.get_flight_info??

Signature:
Server.get_flight_info(
    self,
    context: pyarrow._flight.ServerCallContext,
    descriptor: pyarrow._flight.FlightDescriptor,
) -> pyarrow._flight.FlightInfo
Source:   
    def get_flight_info(
        self, context: flight.ServerCallContext, descriptor: flight.FlightDescriptor
    ) -> flight.FlightInfo:
        """The client can ask for the metadata for a dataset by calling get_flight_info.
        They will use a human-readable FlightDescriptor to describe the dataset they want.
        The FlightDescriptor can either be a path, or a command, but the definition is up to the
        implementer.

        The job of the FlightInfo is to return to the client where the data can be fetched
        from, build a Ticket for the Client to use to ask for the actual data, and provide
        some metadata such as the schema, number of rows, and size.
        """
        dataset_name = descriptor.path[0].decode("utf-8")
        with self._dataset_repo as repo:
            data

We can also use the `FlightDescriptor` to get the schema for the datasetb

In [15]:
schema = client.get_schema(flight.FlightDescriptor.for_path("messages"))
schema

<pyarrow.flight.SchemaResult schema=(id: int64
message_id: string
campaign_id: int64
message_type: string
client_id: int64
channel: string
category: null
platform: string
email_provider: string
stream: string
date: date32[day]
sent_at: timestamp[ms]
is_opened: bool
opened_first_time_at: timestamp[ms]
opened_last_time_at: timestamp[ms]
is_clicked: bool
clicked_first_time_at: timestamp[ms]
clicked_last_time_at: timestamp[ms]
is_unsubscribed: bool
unsubscribed_at: timestamp[ms]
is_hard_bounced: bool
hard_bounced_at: timestamp[ms]
is_soft_bounced: bool
soft_bounced_at: timestamp[ms]
is_complained: bool
complained_at: timestamp[ms]
is_blocked: bool
blocked_at: timestamp[ms]
is_purchased: bool
purchased_at: timestamp[ms]
created_at: timestamp[ns]
updated_at: timestamp[ns])>

In [16]:
Server.get_schema??

Signature:
Server.get_schema(
    self,
    context: pyarrow._flight.ServerCallContext,
    descriptor: pyarrow._flight.FlightDescriptor,
) -> pyarrow._flight.SchemaResult
Source:   
    def get_schema(
        self, context: flight.ServerCallContext, descriptor: flight.FlightDescriptor
    ) -> flight.SchemaResult:
        """Get the schema of the dataset."""
        dataset_name = descriptor.path[0].decode("utf-8")
        with self._dataset_repo as repo:
            dataset = repo.get_dataset(dataset_name)
        if dataset is None:
            raise flight.FlightServerError(f"{dataset_name} not found")
        schema = pq.read_schema(dataset.location, filesystem=self._fs)
        return flight.SchemaResult(schema)
File:      /app/.venv/lib/python3.14/site-packages/flight_tutorial/flight_server/server.py
Type:      function

## Tickets and Locations

One detail we have left out, is that `FlightInfo` also contains the endpoints our client should use
to actually get the data.

Arrow Flight works in a client/server model, a coordinator/worker model
or a mixture of both.

### Endpoints
Let's take a closer look at the `endpoint` again

In [17]:
info.endpoints

[<pyarrow.flight.FlightEndpoint ticket=<pyarrow.flight.Ticket ticket=b'{"name":"messages","bucket":"events","file_name":"messages.parquet","description":"Contains a list of all messages sent with its statuses and meta info.","file_type":"parquet","num_partitions":2073,"num_rows":10000000,"serialized_size":566439389}'> locations=[<pyarrow.flight.Location b'grpc://0.0.0.0:7000'>] expiration_time=None app_metadata=b''>]

`FlightInfo` will contain a list of `endpoints` which the client can connect
to in order to get the data, and a corresponding `Ticket` and `Location` which the server has sent.

Each endpoint represents a part of the data - if the server sends multiple endpoints, that means we can create a client per endpoint
and download that data in parallel.

In [18]:
info.endpoints[0]

<pyarrow.flight.FlightEndpoint ticket=<pyarrow.flight.Ticket ticket=b'{"name":"messages","bucket":"events","file_name":"messages.parquet","description":"Contains a list of all messages sent with its statuses and meta info.","file_type":"parquet","num_partitions":2073,"num_rows":10000000,"serialized_size":566439389}'> locations=[<pyarrow.flight.Location b'grpc://0.0.0.0:7000'>] expiration_time=None app_metadata=b''>

### The Ticket

The `Ticket` is metadata the server needs to fetch the correct data, and the
implementation of that metadata is up to the implementer, Arrow Flight does not prescribe anything. 

In this case, we've gone with simple JSON, but it
could be a Protobuf message, an Avro message or any other set of bytes.

The client doesn't need to care about the contents of the `Ticket`, the server is in charge of
providing the correct metadata. 

In [19]:
info.endpoints[0].ticket

<pyarrow.flight.Ticket ticket=b'{"name":"messages","bucket":"events","file_name":"messages.parquet","description":"Contains a list of all messages sent with its statuses and meta info.","file_type":"parquet","num_partitions":2073,"num_rows":10000000,"serialized_size":566439389}'>

### Endpoint Location

The `Location` is where the `Ticket` can be used, and here the server can also provide multiple options.

It could be providing different geographical locations, so the client can pick the closest, or it could be simply a set of mirrors.

In [20]:
info.endpoints[0].locations

[<pyarrow.flight.Location b'grpc://0.0.0.0:7000'>]

In this case, we have a single endpoint, with a JSON-based ticket, so a simple client/server setup.

We need to grab the ticket from the endpoint, which we can use to get a reader, which we can use
to stream the data

## Do_get - fetch the data

To actually read the data, we call `do_get` with a `Ticket`, which returns a reader, representing a stream of data.
We can then choose to `read_chunk`s from the server or just `read_all`.

Here we call `read_chunk` to grab the next batch from the server.

In [49]:
reader = client.do_get(info.endpoints[0].ticket)

# Stream a chunk of data from the server
chunk = reader.read_chunk()

pl.from_arrow(chunk.data)
# We're not intending to read more data from the reader, so cancel the rest of the request
reader.cancel()

In [50]:
Server.do_get??

Signature:
Server.do_get(
    self,
    context: pyarrow._flight.ServerCallContext,
    ticket: pyarrow._flight.Ticket,
) -> pyarrow._flight.FlightDataStream
Source:   
    def do_get(
        self, context: flight.ServerCallContext, ticket: flight.Ticket
    ) -> flight.FlightDataStream:
        """
        When a client calls get_flight_info, it will get a Ticket which we defined.
        The Ticket describes to the server how to get the data and contains an arbitrary payload
        only meant for the server to understand.
        """
        # We decided on JSON for the ticket payload, so we decode it here.
        dataset = Dataset.model_validate_json(ticket.ticket.decode("utf-8"))

        # Using the ticket payload, we can open the dataset and return a stream of batches.
        table = pq.ParquetFile(dataset.location, filesystem=self._fs, pre_buffer=True)

        def gen():
            try:
                for batch in table.iter_batches(batch_size=256_000):
                  

## Summary of fetching data
![server_client](images/server_client.png)

## Do_Put - uploading data

We can also choose to have our Arrow Flight implement uploading data, using the same streaming mechanism as before, just going the other way

We have some campaign data to go along with our messages, so let's read in the CSV. Unsurprisingly,
Arrow Flight expects the data in Arrow format.

In [23]:
import pyarrow.csv as pc

campaigns = pc.read_csv("data/campaigns.csv")
campaigns

pyarrow.Table
id: int64
campaign_type: string
channel: string
topic: string
started_at: timestamp[ns]
finished_at: timestamp[s]
total_count: int64
ab_test: bool
warmup_mode: bool
hour_limit: int64
subject_length: double
subject_with_personalization: bool
subject_with_deadline: bool
subject_with_emoji: bool
subject_with_bonuses: bool
subject_with_discount: bool
subject_with_saleout: bool
is_test: bool
position: int64
----
id: [[63,64,78,79,89,...,179,35,57,56,237]]
campaign_type: [["bulk","bulk","bulk","bulk","bulk",...,"transactional","transactional","transactional","transactional","transactional"]]
channel: [["mobile_push","mobile_push","mobile_push","mobile_push","mobile_push",...,"email","email","email","email","email"]]
topic: [["sale out","sale out","sale out","sale out","",...,"profile updated","order reminder","order reminder","order reminder","wish list status"]]
started_at: [[2021-04-30 07:22:36.615023000,2021-04-30 09:02:50.817227000,2021-05-06 07:14:10.533318000,2021-05-06 0

Next, we create a FlightDescriptor for the path we want to store the data in, as well as the schema
 of the data. Since Arrow Flight is a streaming the data, it needs to know the schema upfront.

Arrow Flight also uses the bidirectional streaming features of GRPC, so we have both a reader and
a writer as the return value

In [24]:
# Typehinting in Pyarrow library is still work in progress, best to help it along sometimes
writer: flight.FlightStreamWriter
reader: flight.FlightMetadataReader
writer, reader = client.do_put(
    flight.FlightDescriptor.for_path("campaigns"), campaigns.schema
)

In [25]:
# Since we're sending a GRPC stream, we need to tell the server that we're done writing before we can start reading
writer.write_table(campaigns)
writer.done_writing()

In [26]:
result = reader.read()
writer.close()

In [27]:
result.to_pybytes().decode("utf-8")

'Wrote 1907 rows to events/campaigns.parquet'

In [28]:
Server.do_put??

Signature:
Server.do_put(
    self,
    context: pyarrow._flight.ServerCallContext,
    descriptor: pyarrow._flight.FlightDescriptor,
    reader: pyarrow._flight.FlightStreamReader,
    writer: pyarrow._flight.FlightMetadataWriter,
)
Source:   
    def do_put(
        self,
        context: flight.ServerCallContext,
        descriptor: flight.FlightDescriptor,
        reader: flight.FlightStreamReader,
        writer: flight.FlightMetadataWriter,
    ):
        """Do_put is responsible for writing the data to storage. The client will send an
        Arrow Table, and the server will write it to storage. It can also send metadata back
        to the client, such as the number of rows written.
        """
        dataset_name = descriptor.path[0].decode("utf-8")

        location = f"{self._bucket_name}/{dataset_name}.parquet"

        if self._fs.get_file_info(location).type != FileType.NotFound:
            raise flight.FlightServerError(f"{dataset_name} already exists")

        with p

Now that it has been created, we can check the `FlightInfo` to see that it has been correctly registered

In [29]:
info = client.get_flight_info(flight.FlightDescriptor.for_path("campaigns"))

In [30]:
info

<pyarrow.flight.FlightInfo schema=id: int64
campaign_type: string
channel: string
topic: string
started_at: timestamp[ns]
finished_at: timestamp[ms]
total_count: int64
ab_test: bool
warmup_mode: bool
hour_limit: int64
subject_length: double
subject_with_personalization: bool
subject_with_deadline: bool
subject_with_emoji: bool
subject_with_bonuses: bool
subject_with_discount: bool
subject_with_saleout: bool
is_test: bool
position: int64 descriptor=<pyarrow.flight.FlightDescriptor path=[b'campaigns']> endpoints=[<pyarrow.flight.FlightEndpoint ticket=<pyarrow.flight.Ticket ticket=b'{"name":"campaigns","bucket":"events","file_name":"campaigns.parquet","description":null,"file_type":"parquet","num_partitions":1,"num_rows":1907,"serialized_size":61888}'> locations=[<pyarrow.flight.Location b'grpc://0.0.0.0:7000'>] expiration_time=None app_metadata=b''>] total_records=1907 total_bytes=61888 ordered=False app_metadata=b'{"description": null}'>

And we can of course fetch it

In [31]:
reader: flight.FlightStreamReader = client.do_get(info.endpoints[0].ticket)

In [32]:
pl.from_arrow(reader.read_all())

id,campaign_type,channel,topic,started_at,finished_at,total_count,ab_test,warmup_mode,hour_limit,subject_length,subject_with_personalization,subject_with_deadline,subject_with_emoji,subject_with_bonuses,subject_with_discount,subject_with_saleout,is_test,position
i64,str,str,str,datetime[ns],datetime[ms],i64,bool,bool,i64,f64,bool,bool,bool,bool,bool,bool,bool,i64
63,"""bulk""","""mobile_push""","""sale out""",2021-04-30 07:22:36.615023,2021-04-30 07:23:41,48211,null,false,null,146.0,false,false,true,false,false,false,null,null
64,"""bulk""","""mobile_push""","""sale out""",2021-04-30 09:02:50.817227,2021-04-30 09:04:08,1037337,null,false,null,97.0,false,false,true,false,false,false,null,null
78,"""bulk""","""mobile_push""","""sale out""",2021-05-06 07:14:10.533318,2021-05-06 07:15:17,70080,null,false,null,146.0,false,false,true,false,false,false,null,null
79,"""bulk""","""mobile_push""","""sale out""",2021-05-06 09:03:56.486750,2021-05-06 09:42:15,921838,null,false,null,97.0,false,false,true,false,false,false,null,null
89,"""bulk""","""mobile_push""","""""",2021-05-07 11:54:06.168664,2021-05-07 11:54:38,45503,null,false,null,109.0,false,true,true,false,false,false,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
179,"""transactional""","""email""","""profile updated""",null,null,null,null,null,null,134.0,false,false,false,false,false,false,null,null
35,"""transactional""","""email""","""order reminder""",null,null,null,null,null,null,88.0,false,false,false,false,false,false,null,null
57,"""transactional""","""email""","""order reminder""",null,null,null,null,null,null,88.0,false,false,false,false,false,false,null,null


# More bonus stuff - Actions

Arrow Flight has a concept of registered `Actions` - think of them as arbitrary Commands the server
has implemented. We can use these to perform administrative tasks, such as updating a dataset description, or setting permissions.

We can have as many actions as we want, and the server can choose to implement them however it wants.

Flight includes native discoverability for actions, so let's use that to see what we can do.

In [33]:
import json
import pprint

for action in client.list_actions():
    print(f"Action Type: {action.type}")
    pprint.pprint(json.loads(action.description))
    print()

Action Type: update_description
{'description': 'Update a dataset description',
 'schema': {'properties': {'description': {'title': 'Description',
                                           'type': 'string'},
                           'name': {'title': 'Name', 'type': 'string'}},
            'required': ['name', 'description'],
            'title': 'UpdateDatasetRequest',
            'type': 'object'}}

Action Type: delete_dataset
{'description': 'Delete a dataset',
 'schema': {'properties': {'name': {'title': 'Name', 'type': 'string'}},
            'required': ['name'],
            'title': 'DeleteDatasetRequest',
            'type': 'object'}}



In [34]:
Server.list_actions??

Signature: Server.list_actions(self, context: pyarrow._flight.ServerCallContext) -> Iterator[pyarrow._flight.ActionType]
Source:   
    def list_actions(
        self, context: flight.ServerCallContext
    ) -> Iterator[flight.ActionType]:
        """Flight has native support for actions.
        Actions are a way to perform arbitrary operations, and list_actions is a discovery
        mechanism for the client to find out what actions are available.
        """
        actions = [
            (
                "update_description",
                json.dumps(
                    {
                        "description": "Update a dataset description",
                        "schema": UpdateDatasetRequest.model_json_schema(),
                    }
                ),
            ),
            (
                "delete_dataset",
                json.dumps(
                    {
                        "description": "Delete a dataset",
                        "schema": DeleteDatasetReque

The server is advertising an `update_description` action along with a description of the action.

To demonstrate, let's fix the description, since it's not currently very good
An Action consists of an `action_type` and a body of bytes, which are then parsed by the server.

In our case, we have our UpdateDescription Pydantic model contract shared between Client and Server that is expected from the server

In [35]:
from flight_tutorial.flight_server.models import UpdateDatasetRequest

dataset = UpdateDatasetRequest(
    name="campaigns",
    description="Data about the various marketing campaigns",
)

In [36]:
action = flight.Action(
    action_type="update_description", buf=dataset.model_dump_json().encode("utf-8")
)
result = client.do_action(action)

The return value is a generator of `flight.Result`, which is a wrapper around some bytes, so we
need to serialize it to a string to see the result.

In [37]:
next(result).body.to_pybytes().decode("utf-8")

'Updated dataset campaigns description'

In [38]:
client.get_flight_info(flight.FlightDescriptor.for_path("campaigns")).app_metadata

b'{"description": "Data about the various marketing campaigns"}'

In [39]:
Server.do_action??

Signature:
Server.do_action(
    self,
    context: pyarrow._flight.ServerCallContext,
    action: pyarrow._flight.Action,
) -> Iterator[bytes]
Source:   
    def do_action(
        self, context: flight.ServerCallContext, action: flight.Action
    ) -> Iterator[bytes]:
        """When the client wants to perform an action, it will send an Action via do_action.
        What that action does is completely up to the implementation.
        """
        match action.type:
            case "update_description":
                request = UpdateDatasetRequest.model_validate_json(
                    action.body.to_pybytes().decode("utf-8")
                )
                with self._dataset_repo as repo:
                    repo.update_dataset(request)
                yield f"Updated dataset {request.name} description".encode("utf-8")
            case "delete_dataset":
                request = DeleteDatasetRequest.model_validate_json(
                    action.body.to_pybytes().decode("utf

# Do_exchange - Request/Response in the same call

The `do_put` and `do_get` are one-way - data is either coming from the server or being sent to the server. 

`do_exchange` is when we have some data we want the server to process and return to us.

In this example, we can send some message data to the server to calculate a metric, and return that to us. 

As an example, we select a subset of the data, and then we ask the server to do the calculation for us.

In [40]:
info = client.get_flight_info(flight.FlightDescriptor.for_path("messages"))
table = client.do_get(info.endpoints[0].ticket).read_all()
df = pl.from_arrow(table)

To simulate getting a new dataset, we select out the "bulk" message type

In [41]:
bulk = df.filter(pl.col("message_type") == "bulk").to_arrow()

Now we're ready to do our exchange.

In [42]:
# Do_exchange returns a writer and reader, similar to do_put.
# For the FlightDescriptor, we're demonstrating a command, which must be serialized to bytes
command = json.dumps({"metric": "ctr"}).encode("utf-8")
writer, reader = client.do_exchange(flight.FlightDescriptor.for_command(command))

# Since this is a stream, we need to describe the schema of what's to come.
writer.begin(bulk.schema)
# Send the data
writer.write_table(bulk)

# Tell the server we're done writing - ready for the response
writer.done_writing()

# Read the response
result = reader.read_all()

In [43]:
pl.from_arrow(result)

click_rate,sample_ctr
f64,f64
0.023133,0.01225


In [44]:
Server.do_exchange??

Signature:
Server.do_exchange(
    self,
    context: pyarrow._flight.ServerCallContext,
    descriptor: pyarrow._flight.FlightDescriptor,
    reader: pyarrow._flight.FlightStreamReader,
    writer: pyarrow._flight.FlightStreamWriter,
)
Source:   
    def do_exchange(
        self,
        context: flight.ServerCallContext,
        descriptor: flight.FlightDescriptor,
        reader: flight.FlightStreamReader,
        writer: flight.FlightStreamWriter,
    ):
        """Flight can receive and send data within the same call using do_exchange.
        This is commonly used for enriching data, such as calling an ML model to enrich the data,
        or calculate some metric on the data.
        """
        cmd: str = descriptor.command.decode("utf-8")
        match cmd:
            case "ctr":
                sample_ctr_data: pa.Table = reader.read_all()
                df = pl.DataFrame(sample_ctr_data)
                min_date = cast(dt.date, df["date"].min())
                max_date = 

# Conclusion

Apache Arrow is a powerful protocol to have in our back pockets. Whenever you see the need for a client/server implementation, that requires working with large amounts of data, or that would benefit from a RPC-style implementation, reach for Arrow Flight.

It is multi-lingual, supports a distributed architecture natively and allows you to be end-to-end Arrow. It will always be better than ODBC/JSON serialization for large datasets!

In this short demo, we have looked at the basic mechanisms of using Arrow Flight, but these building blocks are something you can use to scale to almost any usecase

